# German ASR fine-tuning (Nixi AI take-home, Task A)

Kaggle notebook: data checks, frozen decode harness, baselines, fine-tuning, final evals.
Training/decoding side of the submission; scoring happens offline in the repo, see README.

Needs: my private Kaggle dataset `nixi-cv26-de-subset` attached as input, GPU (T4), internet on, and an HF write token as Kaggle secret `HF_TOKEN`. Scoring happens outside this notebook (see `run_eval.py` in the repo) - this notebook only produces raw prediction JSONLs and pushes checkpoints to the Hub.
The private dataset is rebuildable via local/scripts.

Ran interactively over a few sessions, so training cells clean up leftover models before loading their own (safe to re-run), and there is a reload-from-Hub cell in case a session dies between training and eval (which happened once).

Note: after cleaning up and re-organizing the notebook cells for submission, I did not rerun the training cells so that I do not lose any of my output results. That's why the cell numbers might be out of order.

In [1]:
!pip install -q jiwer

# the notebook's only environment modification is pip install jiwer — everything else is Kaggle's stock image.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.6 MB/s eta 0:00:00a 0:00:01


In [2]:
# environment + data sanity check
import csv, json, time
import pandas as pd
import librosa
import torch
from pathlib import Path

# note: Kaggle mounts datasets under /kaggle/input/datasets/<user>/<name> these days,
# not the classic /kaggle/input/<name>
DATA = Path("/kaggle/input/datasets/hazalkarakus/nixi-cv26-de-subset")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - fix settings!")
print("clips:", len(list((DATA/"clips").glob("*.mp3"))))
print("subsets:", sorted(p.name for p in (DATA/"subsets").iterdir()))

# the two splits this notebook touches. test is decoded exactly twice per model
# (before + after fine-tuning), dev is used for training decisions and one investigation
test = pd.read_csv(DATA/"subsets/test_subset.tsv", sep="\t",
                   quoting=csv.QUOTE_NONE, dtype=str, na_filter=False)
dev_df = pd.read_csv(DATA/"subsets/dev_subset.tsv", sep="\t",
                     quoting=csv.QUOTE_NONE, dtype=str, na_filter=False)
print(f"test rows: {len(test)}, dev rows: {len(dev_df)}")

GPU: Tesla T4
clips: 12000
subsets: ['dev_subset.tsv', 'test_ids.txt', 'test_subset.tsv', 'train_subset.tsv']
test rows: 1000, dev rows: 1000


## Frozen evaluation harness

Defined once, called for all four evals (tiny/small x base/fine-tuned) plus the dev investigation. Never edited after the first baseline run.

In [3]:
BATCH = 16

@torch.no_grad()   # inference only - no gradients needed, saves memory
def decode_split(model, processor, df, out_path, sr=16_000):
    """
    My frozen evaluation harness. Every model (baseline or fine-tuned) goes
    through this exact function, so WER differences can only come from the
    model - never from a changed setting.

    Deliberately does NOT normalize or score anything: it writes raw
    hypotheses to a JSONL file. Scoring happens in a separate CPU-side
    script, so I can re-score under different normalization rules without
    re-running the GPU, and reviewers can reproduce my numbers from the
    JSONL alone.
    """
    model.eval()
    # Whisper checkpoints ship with baked-in generation constraints that can
    # conflict with explicit language settings on newer transformers.
    # I clear them and set language/task explicitly, so decoding behaviour
    # is identical and visible for all four evals.
    model.generation_config.forced_decoder_ids = None
    model.generation_config.language = "de"
    model.generation_config.task = "transcribe"

    # Fixed sort order -> the same clips in the same order every run.
    rows = df.sort_values("path").to_dict("records")
    start = time.time()
    with open(out_path, "w", encoding="utf-8") as f:
        for i in range(0, len(rows), BATCH):
            batch = rows[i:i+BATCH]
            # librosa decodes the MP3 and resamples 48kHz -> 16kHz,
            # which is the sample rate Whisper was trained on.
            audio = [librosa.load(DATA/"clips"/r["path"], sr=sr)[0] for r in batch]
            feats = processor(audio, sampling_rate=sr, return_tensors="pt",
                              return_attention_mask=True)
            # Greedy decoding (no beam search) - the simplest setting,
            # held constant across all evals. attention_mask tells the model
            # where each clip ends within the padded batch.
            ids = model.generate(feats.input_features.to("cuda"),
                                 attention_mask=feats.attention_mask.to("cuda"))
            hyps = processor.batch_decode(ids, skip_special_tokens=True)
            # One JSON line per clip: which file, what the reference says,
            # what the model said. Raw text, untouched.
            for r, h in zip(batch, hyps):
                f.write(json.dumps({"path": r["path"], "ref": r["sentence"],
                                    "hyp": h.strip()}, ensure_ascii=False) + "\n")
            if (i // BATCH) % 10 == 0:
                print(f"  {i+len(batch)}/{len(rows)} ({time.time()-start:.0f}s)")
    print(f"Wrote {out_path} in {(time.time()-start)/60:.1f} min")

## Baselines (before fine-tuning)

In [4]:
# eval 1/4: whisper-tiny baseline
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to("cuda")

decode_split(model, processor, test, "/kaggle/working/tiny_base.jsonl")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


  16/1000 (15s)
  176/1000 (24s)
  336/1000 (33s)
  496/1000 (42s)
  656/1000 (54s)
  816/1000 (64s)
  976/1000 (72s)
Wrote /kaggle/working/tiny_base.jsonl in 1.2 min


In [5]:
# eval 2/4: whisper-small baseline
# Free tiny's GPU memory before loading the bigger model
del model
torch.cuda.empty_cache()

# Same harness, second model: whisper-small (244M) baseline.
# This is the "already decent at German" end of my comparison.
processor_s = WhisperProcessor.from_pretrained("openai/whisper-small")
model_s = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")

decode_split(model_s, processor_s, test, "/kaggle/working/small_base.jsonl")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

  16/1000 (3s)
  176/1000 (35s)
  336/1000 (70s)
  496/1000 (104s)
  656/1000 (138s)
  816/1000 (173s)
  976/1000 (207s)
Wrote /kaggle/working/small_base.jsonl in 3.5 min


## Fine-tune whisper-tiny

In [13]:
# TRAINING for tiny

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import numpy as np
from dataclasses import dataclass
from datasets import Dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
import jiwer

login(UserSecretsClient().get_secret("HF_TOKEN"))

HUB_REPO = "hazal-karakus/whisper-tiny-de-cv26"

# Free any earlier models if this session still holds them
# and reload a FRESH tiny to fine-tune
# (the smoke-test instance already exists, but reloading guarantees a clean start)
import gc
for name in ["model_s", "trainer"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
# (Including trainer in the cleanup also protects you if you ever re-run the cell after a partial training 
# — otherwise the old trainer would keep a stale model pinned in GPU memory alongside the fresh one.)

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny",
                                             language="german", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to("cuda")
model.generation_config.forced_decoder_ids = None
model.generation_config.language = "de"
model.generation_config.task = "transcribe"

# Load my train/dev subsets. Targets are the RAW sentences (casing,
# punctuation kept) - Whisper natively outputs both, and all normalization
# lives in eval only. Written decision from my plan.
train_df = pd.read_csv(DATA/"subsets/train_subset.tsv", sep="\t",
                       quoting=csv.QUOTE_NONE, dtype=str, na_filter=False)
dev_df   = pd.read_csv(DATA/"subsets/dev_subset.tsv", sep="\t",
                       quoting=csv.QUOTE_NONE, dtype=str, na_filter=False)
train_ds = Dataset.from_pandas(train_df[["path", "sentence"]])
dev_ds   = Dataset.from_pandas(dev_df[["path", "sentence"]])

@dataclass
class OnTheFlyCollator:
    """
    Loads audio and builds model inputs per batch, during training.
    I do this on the fly instead of pre-computing features because 10k
    pre-computed spectrograms would be ~10 GB of cache on Kaggle's limited
    disk - and the CPU keeps up fine while the GPU trains.
    """
    def __call__(self, rows):
        audio = [librosa.load(DATA/"clips"/r["path"], sr=16_000)[0] for r in rows]
        batch = processor.feature_extractor(audio, sampling_rate=16_000,
                                            return_tensors="pt")
        labels = processor.tokenizer([r["sentence"] for r in rows],
                                     return_tensors="pt", padding=True)
        # Padding tokens -> -100 so they're ignored by the loss
        lab = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
        # Trainer adds the start token itself; drop it here to avoid doubling
        if (lab[:, 0] == model.config.decoder_start_token_id).all():
            lab = lab[:, 1:]
        batch["labels"] = lab
        return batch

basic = BasicTextNormalizer()
def compute_metrics(pred):
    """Dev WER for checkpoint selection. Dev is my decision set - test is
    never touched during training. Same Basic normalization as my scorer,
    numbers rule omitted (a checkpoint-picking signal, not a reported number)."""
    ids = pred.predictions
    label_ids = np.where(pred.label_ids != -100, pred.label_ids,
                         processor.tokenizer.pad_token_id)
    hyps = processor.batch_decode(ids, skip_special_tokens=True)
    refs = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": jiwer.wer([basic(r) for r in refs], [basic(h) for h in hyps])}

args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/tiny_ft",
    max_steps=1500,                      # ~2.4 epochs over 10k clips
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,       # effective batch 16, per plan
    learning_rate=1e-5,                  # dev-tuned only if changed
    warmup_steps=100,
    fp16=True,                           # T4 Tensor Cores
    eval_strategy="steps", eval_steps=500,      # dev WER every 500 steps
    save_strategy="steps", save_steps=500, save_total_limit=2,
    load_best_model_at_end=True,         # keep the best-on-dev checkpoint,
    metric_for_best_model="wer",         # not necessarily the last one -
    greater_is_better=False,             # my overfitting guard
    predict_with_generate=True, generation_max_length=225,
    per_device_eval_batch_size=16,
    logging_steps=100,
    report_to="none",
    push_to_hub=False,                   # I push manually below, with the processor
    dataloader_num_workers=2,
    remove_unused_columns=False,   # my dataset holds path+sentence only;
                                   # the collator builds model inputs from
                                   # them on the fly, so nothing may be dropped
)

trainer = Seq2SeqTrainer(model=model, args=args,
                         train_dataset=train_ds, eval_dataset=dev_ds,
                         data_collator=OnTheFlyCollator(),
                         compute_metrics=compute_metrics)
trainer.train()

# Push model AND processor - a model without its processor isn't reloadable.
model.push_to_hub(HUB_REPO, private=True)
processor.push_to_hub(HUB_REPO, private=True)
print("pushed:", HUB_REPO)

# Expected: ~20–40 minutes, with dev WER printed at steps 500/1000/1500

# Result when I ran it: dev WER 28.4 / 27.7 / 28.0 at steps 500/1000/1500,
# so the guard kept the step-1000 checkpoint. Training longer did not help.

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer
500,1.615093,0.566269,0.284439
1000,1.100221,0.552534,0.277449
1500,0.940850,0.552184,0.279923


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

pushed: hazal-karakus/whisper-tiny-de-cv26


In [7]:
# eval 3/4: fine-tuned tiny
# model = best-on-dev checkpoint (step 1000), restored by load_best_model_at_end
decode_split(model, processor, test, "/kaggle/working/tiny_ft.jsonl")

NameError: name 'model' is not defined

In [8]:
# ALTERNATIVE to the cell above, used once for real: my session expired between
# training and eval, so this reloads the pushed checkpoint from the Hub instead
# of retraining. Also the path for anyone verifying my models without a GPU day.
# (skip if the training cell just ran in this session)

# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# from transformers import WhisperProcessor, WhisperForConditionalGeneration
# login(UserSecretsClient().get_secret("HF_TOKEN"))
# REPO = "hazal-karakus/whisper-tiny-de-cv26"
# processor = WhisperProcessor.from_pretrained(REPO)
# model = WhisperForConditionalGeneration.from_pretrained(REPO).to("cuda")
# decode_split(model, processor, test, "/kaggle/working/tiny_ft.jsonl")

processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

  16/1000 (1s)
  176/1000 (9s)
  336/1000 (18s)
  496/1000 (26s)
  656/1000 (35s)
  816/1000 (46s)
  976/1000 (55s)
Wrote /kaggle/working/tiny_ft.jsonl in 0.9 min


In [9]:
# Investigation (not one of the 4 reported evals): tiny_ft scored 33.7 on test
# but 27.7 on dev during training, and I wanted to know why before trusting it.
# If dev is simply an easier speaker sample, tiny BASE should show a similar
# dev-vs-test offset. Dev is my decision set - measuring baselines on it is
# fair game; test stays at two touches per model.
# Outcome: base scored 34.8 on dev vs 46.9 on test -> dev is just easier, my
# test forecast was wrong, not the measurement. Bonus: my harness and the
# trainer's internal eval both said 27.7 for the same model on the same dev
# set - two independent code paths agreeing.

decode_split(model, processor, dev_df, "/kaggle/working/tiny_ft_DEV.jsonl")

# fresh baseline tiny for the comparison
proc_b = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model_b = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to("cuda")
decode_split(model_b, proc_b, dev_df, "/kaggle/working/tiny_base_DEV.jsonl")

  16/1000 (1s)
  176/1000 (9s)
  336/1000 (19s)
  496/1000 (28s)
  656/1000 (37s)
  816/1000 (46s)
  976/1000 (56s)
Wrote /kaggle/working/tiny_ft_DEV.jsonl in 1.0 min


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

  16/1000 (1s)
  176/1000 (10s)
  336/1000 (19s)
  496/1000 (27s)
  656/1000 (35s)
  816/1000 (43s)
  976/1000 (52s)
Wrote /kaggle/working/tiny_base_DEV.jsonl in 0.9 min


## Fine-tune whisper-small

In [18]:
# TRAINING for small

# Differences vs tiny, all deliberate: 500 steps (my first 1500-step attempt
# crashed near the end, but its logged dev trajectory already showed WER flat
# from step 500 to 1000 - 10.98 vs 11.09, within noise for n=1000 - with val
# loss rising 0.229 -> 0.239, so there was no evidence of value past 500),
# LR 5e-6, per-device batch 4 (batch 8 can OOM a T4 with a 244M model),
# accumulation 4 to keep effective batch = 4 x 2 GPUs x 4 = 32, same as tiny,
# eval every 250 steps as the guard inside the shorter run, eval batch 8.

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import numpy as np
from dataclasses import dataclass
from datasets import Dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
import jiwer

login(UserSecretsClient().get_secret("HF_TOKEN"))

HUB_REPO = "hazal-karakus/whisper-small-de-cv26"
MODEL_NAME = "openai/whisper-small"

# Free any earlier models if this session still holds them
# and reload a FRESH tiny to fine-tune
# (the smoke-test instance already exists, but reloading guarantees a clean start)
import gc
for name in ["model", "model_s", "model_b", "trainer"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
# (Including trainer in the cleanup also protects you if you ever re-run the cell after a partial training 
# — otherwise the old trainer would keep a stale model pinned in GPU memory alongside the fresh one.)

processor = WhisperProcessor.from_pretrained(MODEL_NAME,
                                             language="german", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to("cuda")
model.generation_config.forced_decoder_ids = None
model.generation_config.language = "de"
model.generation_config.task = "transcribe"

# Load my train/dev subsets. Targets are the RAW sentences (casing,
# punctuation kept) - Whisper natively outputs both, and all normalization
# lives in eval only. Written decision from my plan.
train_df = pd.read_csv(DATA/"subsets/train_subset.tsv", sep="\t",
                       quoting=csv.QUOTE_NONE, dtype=str, na_filter=False)
dev_df   = pd.read_csv(DATA/"subsets/dev_subset.tsv", sep="\t",
                       quoting=csv.QUOTE_NONE, dtype=str, na_filter=False)
train_ds = Dataset.from_pandas(train_df[["path", "sentence"]])
dev_ds   = Dataset.from_pandas(dev_df[["path", "sentence"]])

@dataclass
class OnTheFlyCollator:
    """
    Loads audio and builds model inputs per batch, during training.
    I do this on the fly instead of pre-computing features because 10k
    pre-computed spectrograms would be ~10 GB of cache on Kaggle's limited
    disk - and the CPU keeps up fine while the GPU trains.
    """
    def __call__(self, rows):
        audio = [librosa.load(DATA/"clips"/r["path"], sr=16_000)[0] for r in rows]
        batch = processor.feature_extractor(audio, sampling_rate=16_000,
                                            return_tensors="pt")
        labels = processor.tokenizer([r["sentence"] for r in rows],
                                     return_tensors="pt", padding=True)
        # Padding tokens -> -100 so they're ignored by the loss
        lab = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
        # Trainer adds the start token itself; drop it here to avoid doubling
        if (lab[:, 0] == model.config.decoder_start_token_id).all():
            lab = lab[:, 1:]
        batch["labels"] = lab
        return batch

basic = BasicTextNormalizer()
def compute_metrics(pred):
    """Dev WER for checkpoint selection. Dev is my decision set - test is
    never touched during training. Same Basic normalization as my scorer,
    numbers rule omitted (a checkpoint-picking signal, not a reported number)."""
    ids = pred.predictions
    label_ids = np.where(pred.label_ids != -100, pred.label_ids,
                         processor.tokenizer.pad_token_id)
    hyps = processor.batch_decode(ids, skip_special_tokens=True)
    refs = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": jiwer.wer([basic(r) for r in refs], [basic(h) for h in hyps])}

args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/small_ft",
    max_steps=500,                       # crashed 1st run's dev trajectory: WER flat 500->1000
                                         # (10.98 vs 11.09, within noise), val loss rose
                                         # (0.229->0.239) -> no evidence of value past 500
    per_device_train_batch_size=4,       # 244M model on a T4: batch 8 can OOM
    gradient_accumulation_steps=4,       # keeps effective batch = 4 x 2 GPUs x 4 = 32, same as tiny's run
    learning_rate=5e-6,                  # per plan: gentler for the bigger model
    warmup_steps=100,
    fp16=True,                           # T4 Tensor Cores
    eval_strategy="steps", eval_steps=250,      # two dev checkpoints within the shorter run
    save_strategy="steps", save_steps=250, save_total_limit=2,
    load_best_model_at_end=True,         # keep the best-on-dev checkpoint,
    metric_for_best_model="wer",         # not necessarily the last one -
    greater_is_better=False,             # my overfitting guard
    predict_with_generate=True, generation_max_length=225,
    per_device_eval_batch_size=8,        # was 16 tiny; small's activations during generation are much larger
    logging_steps=100,
    report_to="none",
    push_to_hub=False,                   # I push manually below, with the processor
    dataloader_num_workers=2,
    remove_unused_columns=False,   # my dataset holds path+sentence only;
                                   # the collator builds model inputs from
                                   # them on the fly, so nothing may be dropped
)

trainer = Seq2SeqTrainer(model=model, args=args,
                         train_dataset=train_ds, eval_dataset=dev_ds,
                         data_collator=OnTheFlyCollator(),
                         compute_metrics=compute_metrics)
trainer.train()

# Push model AND processor - a model without its processor isn't reloadable.
model.push_to_hub(HUB_REPO, private=True)
processor.push_to_hub(HUB_REPO, private=True)
print("pushed:", HUB_REPO)

# Expected: ~1.5–2.5 hours, with dev WER printed at steps 500/1000/1500
# Prediction time, on the record before the run: my band for small_ft is 12–17% 
# — centered on "modest improvement, maybe ~1–3 points," with the honest possibility of zero or slightly worse. 
# My hypothesis says the delta shrinks; 
# the paired bootstrap afterward will tell us whether whatever delta appears is even distinguishable from noise.

# Result when I ran it: dev WER 11.26 at step 250, 10.79 at step 500.
# (Slightly better than the crashed run's step-500 value because the shorter
# max_steps means the LR schedule finishes its decay by 500.)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


pushed: hazal-karakus/whisper-small-de-cv26


In [7]:
# check the Hub repo is self-sufficient before relying on it: reload the
# processor from the Hub and confirm it tokenizes identically to the local one.
# (the push printed "skipping empty commit" for the processor, so I verified
# instead of assuming - turned out fine, tokenizer.json carries the vocab)

# p_hub = WhisperProcessor.from_pretrained("hazal-karakus/whisper-small-de-cv26")
# s = "Größere Bäume - 25 Stück, natürlich!"
# assert p_hub.tokenizer(s).input_ids == processor.tokenizer(s).input_ids, "MISMATCH"
# print("processor roundtrip OK")

In [20]:
# eval 4/4: fine-tuned small
# model = best-on-dev checkpoint (step 500), restored by load_best_model_at_end
decode_split(model, processor, test, "/kaggle/working/small_ft.jsonl")

  16/1000 (5s)
  176/1000 (39s)
  336/1000 (76s)
  496/1000 (111s)
  656/1000 (147s)
  816/1000 (183s)
  976/1000 (220s)
Wrote /kaggle/working/small_ft.jsonl in 3.8 min


In [10]:
# environment record for the repo
!python --version
!pip freeze | grep -iE "^(torch|transformers|datasets|librosa|jiwer|numpy|pandas|accelerate)="
!nvidia-smi --query-gpu=name --format=csv,noheader

Python 3.12.13
accelerate==1.13.0
datasets==5.0.0
jiwer==4.0.0
librosa==0.11.0
numpy==2.0.2
pandas==2.3.3
torch==2.10.0+cu128
transformers==5.0.0
Tesla T4
Tesla T4
